# Import necessary lib

In [1]:
import numpy as np
import pandas as pd
import warnings
from scipy.stats import randint

# Scikit-learn: preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Scikit-learn: models
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

### Task 1: Data Loading

In [2]:
df = pd.read_csv('stock_data.csv')
df.head(3)

,Unnamed: 0,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5
0,2020-01-01,101.764052,100.160928,99.494642,99.909756,101.761266
1,2020-01-02,102.171269,99.969968,98.682973,100.640755,102.528643
2,2020-01-03,103.171258,99.575237,98.182139,100.574847,101.887811


In [3]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Stock Data Profiling Report")

In [4]:
# Display the profiling report
profile.to_file("stock_data_profiling_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 144.45it/s]


### Task 2: Data Preprocessing

In [5]:
# 1. Drop 'Unnamed: 0' if it exists
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

# 2. Separate features (X) and target (y)
X = df.drop('Stock_2', axis=1)
y = df['Stock_2']

# 3. Identify numerical columns
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

# 4. Numeric pipeline: impute missing values + scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 5. ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features)
    ],
    remainder='drop'  # drop non-numeric columns if any
)

print("Preprocessor ready with numeric features:", numeric_features)


Preprocessor ready with numeric features: ['Stock_1', 'Stock_3', 'Stock_4', 'Stock_5']


### Task 3: Pipeline Creation

In [6]:
# numeric pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   
    ('scaler', StandardScaler())
])

# preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features)
    ]
)

# KNN model 
knn = KNeighborsRegressor(n_neighbors=50)

# full pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', knn)
])


### Data Splitting

In [7]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Task 5: Model Training

In [8]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse 

### Task 6: Cross-Validation

In [9]:

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f"RMSE: {cv_rmse}")
mean_rmse = np.mean(cv_rmse)
print(f"\nAverage RMSE : {mean_rmse:.4f}")
std_rmse = np.std(cv_rmse)
print(f"Standard deviation : {std_rmse:.4f}")

RMSE: [4.55838454 4.97704379 4.13898683 5.02772912 5.31374603]

Average RMSE : 4.8032
Standard deviation : 0.4105


### Task 7: Hyperparameter Tuning

In [10]:
# Build pipeline
knn_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor())
])

# Parameter distributions
param_dist = {
    'model__n_neighbors': randint(5, 100),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2],
    'model__leaf_size': randint(20, 50)
}

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=knn_pipe,
    param_distributions=param_dist,
    n_iter=50,              
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# Fit
random_search.fit(X_train, y_train)



Fitting 5 folds for each of 50 candidates, totalling 250 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__leaf_size': <scipy.stats....001BF61FC2BD0>, 'model__n_neighbors': <scipy.stats....001BF616FAB70>, 'model__p': [1, 2], 'model__weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation 

In [11]:
# Best parameters
print("Best KNN Params:", random_search.best_params_)
print("Best CV R2 Score:", random_search.best_score_)

# Evaluate on test set
y_pred = random_search.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Best KNN Params: {'model__leaf_size': 40, 'model__n_neighbors': 6, 'model__p': 2, 'model__weights': 'distance'}
Best CV R2 Score: 0.9832468778547934
R2 Score: 0.9919
RMSE: 1.0913
MAE: 0.8228


### Task 8: Best Model Selection & Task 9: Model Performance Evaluation

In [12]:
# Get the best pipeline from RandomizedSearchCV
best_model_pipeline = random_search.best_estimator_

# Predict on test set
y_pred_best_model = best_model_pipeline.predict(X_test)

# Evaluate
r2_best_model = r2_score(y_test, y_pred_best_model)
rmse_best_model = np.sqrt(mean_squared_error(y_test, y_pred_best_model))
mae_best_model = mean_absolute_error(y_test, y_pred_best_model)

# Print results
print("\n--- Best Model Performance on Test Set ---")
print("Best Params:", random_search.best_params_)
print(f"R2 Score: {r2_best_model:.4f}")
print(f"RMSE: {rmse_best_model:.4f}")
print(f"MAE: {mae_best_model:.4f}")



--- Best Model Performance on Test Set ---
Best Params: {'model__leaf_size': 40, 'model__n_neighbors': 6, 'model__p': 2, 'model__weights': 'distance'}
R2 Score: 0.9919
RMSE: 1.0913
MAE: 0.8228
